# GameWith 파티 데이터 변환 (Google Colab)

이 노트북은 **GameWith 사이트에 직접 접속하지 않습니다.** PC 크롬에서 `GameWith_PC브라우저_수집.js`로 내려받은 `GameWith_파티_전체데이터.json`을 업로드해 CSV/JSON으로 정리합니다.

- `pokemon.csv`: 파티별 6마리 이름·도구·특성·성격·4기술·HABCDS 투자 포인트·수집 오류 상태
- `teams.csv`: 파티별 시즌·순위·작성자·6마리 이름
- `summary.json`: 수집 건수 및 페이지 이동 상태
- `result.zip`: 위 파일을 하나로 묶은 압축 파일

> 브라우저 추출 결과에는 **실제 전체 페이지를 끝까지 이동했는지 보장할 수 없을 때** 상태가 기록됩니다. 추출된 숫자만으로 전체 수집 성공을 단정하지 마세요.

In [ ]:
from google.colab import files
import json

uploaded = files.upload()
if len(uploaded) != 1:
    raise ValueError("수집 결과 JSON 파일 한 개만 업로드하세요.")
filename, payload = next(iter(uploaded.items()))
data = json.loads(payload.decode('utf-8-sig'))
if not isinstance(data, dict) or 'teams' not in data or 'pokemon' not in data:
    raise ValueError('GameWith_파티_전체데이터.json 형식이 아닙니다.')
print('업로드됨:', filename)
print('파티:', len(data['teams']), '포켓몬:', len(data['pokemon']))
print('수집 상태:', data.get('status', 'UNKNOWN'))

In [ ]:
from pathlib import Path
from collections import Counter
import pandas as pd
import zipfile, json

out=Path('/content/gamewith_processed')
out.mkdir(exist_ok=True, parents=True)

# 엑셀에서 텍스트가 공식으로 오인되는 문제 방지
def safe_cell(value):
    if value is None: return ''
    if isinstance(value, str) and value.lstrip().startswith(('=', '+', '-', '@')):
        return "'" + value
    return value

mon_cols=['page_number','party_index','team_key','season','rank','player','points','slot','pokemon',
          'detail_found','detail_id','item','ability','nature','move_1','move_2','move_3','move_4',
          'hp_pt','atk_pt','def_pt','spa_pt','spd_pt','spe_pt','error']
team_cols=['page_number','party_index','team_key','season','rank','player','points'] + [f'pokemon_{i}' for i in range(1,7)]
mon=[]
for row in data['pokemon']:
    mon.append({k:safe_cell(row.get(k,'')) for k in mon_cols})
teams=[]
for row in data['teams']:
    item={k:safe_cell(row.get(k,'')) for k in team_cols if not k.startswith('pokemon_')}
    item['team_key']=safe_cell(row.get('team_key',''))
    for pokemon in row.get('pokemon',[]):
        s=pokemon.get('slot')
        if isinstance(s, int) and 1<=s<=6:
            item[f'pokemon_{s}']=safe_cell(pokemon.get('pokemon',''))
    teams.append(item)

pd.DataFrame(mon, columns=mon_cols).to_csv(out/'pokemon.csv', index=False, encoding='utf-8-sig')
pd.DataFrame(teams, columns=team_cols).to_csv(out/'teams.csv', index=False, encoding='utf-8-sig')
(out/'pokemon.json').write_text(json.dumps(data['pokemon'], ensure_ascii=False,indent=2),encoding='utf-8')
(out/'teams.json').write_text(json.dumps(data['teams'], ensure_ascii=False,indent=2),encoding='utf-8')
summary={
 'collector_status':data.get('status','UNKNOWN'),
 'total_pages_visited':len(data.get('pages',[])),
 'total_teams':len(data['teams']),
 'pokemon_rows':len(data['pokemon']),
 'detail_found':sum(x.get('detail_found') is True for x in data['pokemon']),
 'detail_missing':sum(x.get('detail_found') is not True for x in data['pokemon']),
 'errors':dict(Counter(str(x.get('error','')) for x in data['pokemon'] if x.get('error'))),
 'pages':[{'page_number':x.get('page_number'),'party_count':x.get('party_count'),'next_page_method':x.get('next_page_method')} for x in data.get('pages',[])],
}
(out/'summary.json').write_text(json.dumps(summary,ensure_ascii=False,indent=2),encoding='utf-8')
archive=out/'result.zip'
with zipfile.ZipFile(archive,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for file in sorted(out.iterdir()):
        if file.is_file() and file!=archive: z.write(file,arcname=file.name)
print(json.dumps(summary,ensure_ascii=False,indent=2))
print('파일 생성 완료:',archive)


In [ ]:
files.download(str(archive))